# Exercise 5 — Max the Builder


## Task

Max Welfare bought a remaining stock of **5,000 nails**, **33 buckets of finish**, and **770 planks** for EUR 500. He can build only fences and gazebos. Determine how many of each product he should build to maximize profit.

| Parameter | Unit | Fence | Gazebo |
|---|---|---:|---:|
| Nails | no. | 350 | 550 |
| Finish | no. | 4.5 | 3.5 |
| Planks | no. | 70 | 140 |
| Profit | EUR | 2000 | 2500 |

Tasks:

1. Formulate the problem algebraically.
2. Translate it into JuMP and solve it.
3. Interpret the solution.

The EUR 500 purchase price is a fixed cost and is therefore subtracted from the objective in the supplied implementation.


## Reference implementation


In [ ]:
using JuMP, HiGHS
using DataFrames

# create dataframe from input table
df = DataFrame(
    Inputs = ["Nails", "Finish", "Planks"],
    Fence = [350, 4.5, 70],
    Gazebo = [550, 3.5, 140],
    Max_Production = [5000, 33, 770],
)

inputs = df.Inputs # production inputs
products = names(df)[2:3] # products

# create dicts
av = Dict(df.Inputs .=> df.Max_Production)
cm = Dict(products .=> [2000, 2500])
f = Dict((i, p) => df[findfirst(x -> x == i, df.Inputs), p] for i in inputs, p in products)

# model
model = Model(HiGHS.Optimizer)
@variable(model, x[products] >= 0) # production variable for each product
@constraint(
    model,
    production_constraint[i=inputs],
    f[i, "Fence"]*x["Fence"] + f[i, "Gazebo"]*x["Gazebo"] <= av[i]
) # max production
@objective(model, Max, sum(x[p]*cm[p] for p in products)-500)
optimize!(model)
objective_value(model)
value.(x)
dual.(production_constraint)
